# Artwork Eval on Colab (No Drive)

This notebook runs the artwork benchmark directly in Colab using git clones under `/content`.

- Benchmark repo: `physical168/kv-compression-benchmark`
- Engine repo: `GabrieleSanmartino/CompressionExperiments`
- Entry point: `CompressionExperiments/experiment_manager/run_compression_image.py`

> No Google Drive is required.

## Step 0 - Runtime

Set runtime to **GPU** in Colab before running.

## Step 0.5 - GitHub web login (popup)

Use GitHub CLI device/web auth so private repo clone can succeed without tokens in notebook cells.

- This opens a browser-based GitHub login flow.
- Re-auth is needed after runtime reset.

In [ ]:
# Install GitHub CLI and authenticate via web/device flow
!apt-get -qq update
!apt-get -qq install gh -y

# Opens browser/device verification flow
!gh auth login -h github.com -p https -w

# quick check
!gh auth status

In [ ]:
from pathlib import Path
import shutil, subprocess

BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
CE_URL = "https://github.com/GabrieleSanmartino/CompressionExperiments.git"

BENCH_DIR = Path("/content/kv-compression-benchmark")
CE_DIR = Path("/content/CompressionExperiments")

# Ensure gh auth is available before cloning private repos
subprocess.check_call(["gh", "auth", "status"])

for p in [BENCH_DIR, CE_DIR]:
    if p.exists():
        shutil.rmtree(p)

subprocess.check_call(["git", "clone", "--depth", "1", BENCH_URL, str(BENCH_DIR)])
subprocess.check_call(["git", "clone", "--depth", "1", CE_URL, str(CE_DIR)])
print("Cloned:", BENCH_DIR)
print("Cloned:", CE_DIR)

In [ ]:
%cd /content/CompressionExperiments
!git submodule update --init --recursive

!pip install -q -U pip
!pip install -q transformers accelerate bitsandbytes pandas pyyaml scikit-learn matplotlib tqdm pillow
!pip install -q -e kvpress

## Step 1 - Prepare artwork dataset

This copies `paintings.csv` and local images from the benchmark repo clone into
`CompressionExperiments/experiment_manager/datasets/artwork/`.

> If your images are not in the benchmark repo, upload them to Colab first and adjust `SRC_IMG_DIR`.

In [ ]:
from pathlib import Path
import shutil

SRC_ART = Path("/content/kv-compression-benchmark/datasets/artwork")
SRC_CSV = SRC_ART / "paintings.csv"
SRC_IMG_DIR = SRC_ART / "images"

DST_ART = Path("/content/CompressionExperiments/experiment_manager/datasets/artwork")
DST_ART.mkdir(parents=True, exist_ok=True)

if not SRC_CSV.is_file():
    raise FileNotFoundError(f"Missing paintings.csv: {SRC_CSV}")
if not SRC_IMG_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing image folder: {SRC_IMG_DIR}. Upload images first or change SRC_IMG_DIR."
    )

shutil.copy2(SRC_CSV, DST_ART / "paintings.csv")
if (DST_ART / "images").exists():
    shutil.rmtree(DST_ART / "images")
shutil.copytree(SRC_IMG_DIR, DST_ART / "images")

print("Dataset ready:", DST_ART)
print("paintings.csv exists:", (DST_ART / "paintings.csv").is_file())
print("images dir exists:", (DST_ART / "images").is_dir())

## Step 2 - Run compression (image)

Default uses `ExpectedAttentionPress` only. Add other presses if needed.

In [ ]:
%cd /content/CompressionExperiments/experiment_manager

!python run_compression_image.py \
  --model llava-hf/llama3-llava-next-8b-hf \
  --press ExpectedAttentionPress \
  --ratios 0.4 0.8 0.95 \
  --max-records 20 \
  --output-dir results \
  --attn-impl sdpa

## Step 3 - Evaluate (P/R/F1)

In [ ]:
%cd /content/CompressionExperiments/experiment_manager
!python evaluate.py --results-dir results --summary

## Step 4 - Optional: Zip results for download

In [ ]:
%cd /content/CompressionExperiments/experiment_manager
!zip -r /content/ce_artwork_results.zip results
print("Created: /content/ce_artwork_results.zip")